# Dubbing 工作 Notebook
由私有 Dubbing_Launcher.ipynb 自动获取并执行。源码与本 Notebook 固定在启动时解析出的同一提交。


In [ ]:
from pathlib import Path
import sys, os, subprocess, uuid, json, tempfile

REPOSITORY = 'https://github.com/Kaitzz/free-dubbing.git'
SOURCE_COMMIT = globals().get('DUBBING_SOURCE_COMMIT', 'ff3ff39c3b7a3d48fbb15079a9f33ce095387801')
# Each release has its own checkout; reruns never overwrite an older runtime.
base = Path('/content/free-dubbing')
base.mkdir(parents=True, exist_ok=True)
root = base / SOURCE_COMMIT
if not root.exists():
    with tempfile.TemporaryDirectory(prefix='checkout-', dir=base) as temporary:
        checkout = Path(temporary) / 'source'
        subprocess.run(['git', 'init', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', REPOSITORY, SOURCE_COMMIT], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
        actual = subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        assert actual == SOURCE_COMMIT, 'Downloaded source version does not match'
        checkout.rename(root)
actual = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT, 'Unexpected existing source version'
subprocess.run(['git', '-C', str(root), 'diff', '--exit-code', 'HEAD', '--'], check=True)
assert (root/'scripts/colab_worker.py').is_file(), 'Worker source missing'
# Share downloaded models across releases and reuse the previous upload-based cache.
old_cache = Path('/content/youdub-pipeline/model-cache')
cache = (old_cache if old_cache.is_dir() else base/'model-cache').resolve()
cache.mkdir(parents=True, exist_ok=True)
os.environ['MODEL_CACHE_DIR'] = str(cache)
os.chdir(root)
print(f'Source ready: Kaitzz/free-dubbing @ {SOURCE_COMMIT[:7]}')


## 安装（首次需要几分钟）
专用 Python 环境复用 Colab 的 CUDA PyTorch，避免给 notebook 内核安装互相冲突的依赖。


In [ ]:
import importlib.metadata as metadata
torch_version = metadata.version('torch')
audio_version = metadata.version('torchaudio')
minor = '.'.join(torch_version.split('+')[0].split('.')[:2])
codec_versions = {'2.5':'0.1', '2.6':'0.2', '2.7':'0.5', '2.8':'0.7', '2.9':'0.9', '2.10':'0.10', '2.11':'0.11'}
assert minor in codec_versions, f'请先更新 TorchCodec 兼容配置：torch={torch_version}'
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'fonts-noto-cjk', 'fontconfig', 'git'], check=True)
venv = root/'.venv-colab'
# virtualenv seeds its own pip and repairs incomplete environments.
setup = subprocess.run([sys.executable, 'scripts/colab_environment.py', str(root)],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(setup.stdout)
setup.check_returncode()
python = str(venv/'bin/python')
constraints = root/'colab-constraints.txt'
constraints.write_text(f'torch=={torch_version}\ntorchaudio=={audio_version}\ntorchcodec=={codec_versions[minor]}.*\n')
subprocess.run([python, '-m', 'pip', 'install', '-c', str(constraints), '-r', 'requirements-colab.txt'], check=True)
subprocess.run([python, '-m', 'pip', 'install', '--upgrade', 'yt-dlp[default]'], check=True)
os.environ['PATH'] = str(venv/'bin') + os.pathsep + os.environ.get('PATH','')
demucs = root/'submodule/demucs'
commit = 'e976d93ecc3865e5757426930257e200846a520a'
if not (demucs/'.git').exists():
    demucs.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'init', str(demucs)], check=True)
subprocess.run(['git', '-C', str(demucs), 'fetch', '--depth', '1', 'https://github.com/facebookresearch/demucs.git', commit], check=True)
subprocess.run(['git', '-C', str(demucs), 'checkout', '--detach', commit], check=True)
print('安装完成；继续运行预检')


In [ ]:
subprocess.run([python, 'scripts/colab_preflight.py'], check=True)


## 连接本机 GUI
固定地址 https://dubbing.corneliazhang.me；连接密码由私有启动壳的 DUBBING_PASSWORD 提供。MiniMax 密钥仍从 MINIMAX_API_KEY Secret 获取。


In [ ]:
from getpass import getpass
from google.colab import userdata
tunnel_url = 'https://dubbing.corneliazhang.me'
worker_token = str(globals().get('DUBBING_PASSWORD', '')).strip()
if not worker_token:
    worker_token = getpass('固定连接密码: ').strip()
if len(worker_token) != 8 or not worker_token.isascii() or not worker_token.isdigit():
    raise ValueError('连接密码应为 8 位数字，与本机 GUI 一致')
try:
    minimax_key = userdata.get('MINIMAX_API_KEY')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    minimax_key = getpass('MiniMax API Key: ')
assert worker_token and minimax_key.strip(), '密钥不能为空'
if 'YOUTUBE_COOKIES' not in globals():
    # Optional: paste the complete Netscape cookies.txt into the Colab Secret.
    from google.colab import userdata
    try:
        YOUTUBE_COOKIES = userdata.get('YOUTUBE_COOKIES')
    except userdata.SecretNotFoundError:
        YOUTUBE_COOKIES = ''
    except userdata.NotebookAccessError as exc:
        raise RuntimeError('请允许此 Notebook 访问 YOUTUBE_COOKIES Secret') from None
worker_env = os.environ.copy()
worker_env.update(YOUDUB_WORKER_TOKEN=worker_token, OPENAI_API_KEY=minimax_key.strip())
worker_env['YOUTUBE_COOKIES'] = YOUTUBE_COOKIES or ''
del worker_token, minimax_key


## 开始领取任务
运行后回到本机网页上传视频；自动模式会连续执行，手动模式会等待网页上的「继续」。本单元默认保持两小时，正在执行的阶段会执行完。停止单元后 Colab 断开；未回传的阶段会在约三分钟后标为失败，可从最后收到的检查点重试。


In [ ]:
worker_process = subprocess.Popen([python, '-u', 'scripts/colab_worker.py', '--url', tunnel_url, '--minutes', '120'],
    env=worker_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in worker_process.stdout:
        print(line, end='', flush=True)
    code = worker_process.wait()
    assert code == 0, f'Colab worker 退出码: {code}'
except KeyboardInterrupt:
    import signal
    worker_process.send_signal(signal.SIGINT)
    try:
        worker_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        worker_process.terminate()
        worker_process.wait()
    raise
